Developing a semantic search engine for retrieving relevant information from machine learning research papers and Kaggle solution writeups.

## Architecture

<div align="center">
    
    PDFs
      ↓
    Text Extraction
      ↓
    Chunking
      ↓
    Embedding Model
      ↓
    Vector Embeddings
      ↓
    FAISS Vector Index
-------------------------
    User Query
      ↓
    Query Embedding
      ↓
    Similarity Search (Semantic Search + Keyword Search(BM25))
      ↓
    Top-k Matching Chunks

</div>

In [1]:
!pip install sentence-transformers faiss-cpu pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 63.5 MB/s eta 0:00:00


## Load pdfs
This extracts:

* filename
* page number
* text

In [2]:
import fitz #PyMuPDF
import os
from pathlib import Path

dataset_path = '/kaggle/input/datasets/adwaittagalpallewar/kaggle-whitepaper-pdfs'

In [3]:
documents = []

pdf_files = list(Path(dataset_path).glob('*.pdf'))

for pdf_file in pdf_files:
    
    doc = fitz.open(pdf_file)

    for page_num in range(len(doc)):

        page = doc[page_num]
        text = page.get_text()

        documents.append({
            'source': pdf_file.name,
            'page': page_num,
            'text': text
        })

print(f'Loaded {len(documents)} pages')

Loaded 260 pages


In [4]:
documents[4]

{'source': '22365_19_Agents_v8.pdf',
 'page': 4,
 'text': 'Agents\n5\nFebruary 2025\nWhat is an agent?\nIn its most fundamental form, a Generative AI agent can be defined as an application that \nattempts to achieve a goal by observing the world and acting upon it using the tools that it \nhas at its disposal. Agents are autonomous and can act independently of human intervention, \nespecially when provided with proper goals or objectives they are meant to achieve. Agents \ncan also be proactive in their approach to reaching their goals. Even in the absence of \nexplicit instruction sets from a human, an agent can reason about what it should do next to \nachieve its ultimate goal. While the notion of agents in AI is quite general and powerful, this \nwhitepaper focuses on the specific types of agents that Generative AI models are capable of \nbuilding at the time of publication.\nIn order to understand the inner workings of an agent, let’s first introduce the foundational \ncomponents t

## Chunking function

In [5]:
def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

### Chunked dataset

In [6]:
chunked_documents = []

for doc in documents:
    chunks = chunk_text(doc['text'])

    for i, chunk in enumerate(chunks):

        if len(chunk.strip()) > 50:

            chunked_documents.append({
                'source': doc['source'],
                'page': doc['page'],
                'chunk_id': i,
                'text': chunk
            })

print(f"Created {len(chunked_documents)} chunks")

Created 905 chunks


In [7]:
chunked_documents[2]

{'source': '22365_19_Agents_v8.pdf',
 'page': 2,
 'chunk_id': 0,
 'text': 'Introduction\x08\n4\nWhat is an agent?\x08\n5\nThe model\x08\n6\nThe tools\x08\n7\nThe orchestration layer\x08\n7\nAgents vs. models\x08\n8\nCognitive architectures: How agents operate \x08\n8\nTools: Our keys to the outside world\x08\n12\nExtensions \x08\n13\nSample Extensions \x08\n15\nFunctions \x08\n18\nUse cases\t\n21\nFunction sample code\t\n24\nData stores\x08\n27\nImplementation and application\t\n28\nTools recap\x08\n32\nEnhancing model performance with targeted learning\x08\n33\nAgent quick start with LangChain\x08\n35\nProduction applications with Vertex AI agents\x08\n3'}

## Embeddings

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
texts = [doc['text'] for doc in chunked_documents]

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

In [10]:
embeddings.shape

(905, 384)

The shape (905, 384) suggests there are 905 chunks and each chunk is represented by a vector of 384 dimensions.


## Retrival
We’ll use cosine similarity.

IMPORTANT:
FAISS uses L2 distance by default.

For cosine similarity:

* normalize embeddings first

### Semantic Search/retrival
understands meaning 

In [11]:
import faiss
import numpy as np

#normalization
faiss.normalize_L2(embeddings)

In [12]:
#creating index
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension) #bascically creating a vector database

Inner Product ≈ Cosine Similarity

In [13]:
#adding embeddigns to index
index.add(embeddings)

print(f"Total vectors in index: {index.ntotal}")

Total vectors in index: 905


In [14]:
def semantic_search(query, top_k=5):

    #embed query
    query_embedding = model.encode([query], convert_to_numpy=True)

    #Normalize query vector
    faiss.normalize_L2(query_embedding)

    #Search index
    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        results.append({
            'score': float(score),
            'source': chunked_documents[idx]['source'],
            'page': chunked_documents[idx]['page'],
            'text': chunked_documents[idx]['text']
        })

    return results

In [15]:
results = semantic_search("model architecture")

In [16]:
for i, result in enumerate(results):
    
    print("=" * 80)
    print(f"Result {i+1}")
    print(f"Score : {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page  : {result['page']}")
    print()
    print(result["text"][:1000])
    print()

Result 1
Score : 0.5760
Source: 22365_19_Agents_v8.pdf
Page  : 5

Agents
6
February 2025
Figure 1. General agent architecture and components
The model
In the scope of an agent, a model refers to the language model (LM) that will be utilized as 
the centralized decision maker for agent processes. The model used by an agent can be one 
or multiple LM’s of any size (small / large) that are capable of following instruction based 
reasoning and logic frameworks, like ReAct, Chain-of-Thought, or Tree-of-Thoughts. Models 
can be general purpose, multimodal or fine-tu

Result 2
Score : 0.5280
Source: whitepaper 1.pdf
Page  : 5

arious architectures and approaches building up to the large language 
models and the architectures being used at the time of publication. It also discusses fine-
We believe that this new crop of 
technologies has the potential to 
assist, complement, empower, 
and inspire people at any time 
across almost any field.


Result 3
Score : 0.5136
Source: whitepaper 1.pdf
Pa

### BM25-retrival/search
understands exact terms

In [17]:
!pip install rank-bm25

In [18]:
from rank_bm25 import BM25Okapi

#BM25 works on tokenized text.
tokenized_chunks = [
    doc['text'].lower().split()
    for doc in chunked_documents
]

bm25 = BM25Okapi(tokenized_chunks) #this is bm25 index/ keyword database

In [19]:
def bm25_search(query, top_k=5):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:

        results.append({
            "score": scores[idx],
            "source": chunked_documents[idx]["source"],
            "page": chunked_documents[idx]["page"],
            "text": chunked_documents[idx]["text"]
        })

    return results

In [20]:
results = bm25_search("transformer architecture")

In [21]:
for i, result in enumerate(results):
    
    print("=" * 80)
    print(f"Result {i+1}")
    print(f"Score : {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page  : {result['page']}")
    print()
    print(result["text"][:1000])
    print()

Result 1
Score : 8.5768
Source: whitepaper 1.pdf
Page  : 8

Foundational Large Language Models & Text Generation
9
February 2025
Herein, we discuss the first version of the transformer model and then move on to the more 
recent advanced models and algorithms.
Transformer
The transformer architecture was developed at Google in 2017 for use in a translation model.1 
It’s a sequence-to-sequence model capable of converting sequences from one domain 
into sequences in another domain. For example, translating French sentences to English 
sentences. The origina

Result 2
Score : 7.5671
Source: whitepaper 1.pdf
Page  : 8

rench sentences to English 
sentences. The original transformer architecture consists of two parts: an encoder and a 
decoder. The encoder converts the input text (e.g., a French sentence) into a representation, 
which is then passed to the decoder. The decoder uses this representation to generate the 
output text (e.g., an English translation) autoregressively.1 Notably, the